# Artifact Evaluation (Config-Driven)

Use one config cell to control exactly what gets evaluated and displayed.

This notebook is optimized for answering:
- Which runs/configurations performed best overall?
- Which runs performed best for a specific subject?
- What compact tables should be exported?

In [10]:
from pathlib import Path
import json
from typing import Any
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## 1. Evaluation Config

Edit only this cell when you want to run a different evaluation setup.

- Set which runs to compare
- Set the primary metric for ranking
- Toggle optional outputs
- Set a focus subject for subject-level analysis

In [11]:
RUN_IDS = [
  "20260401_0944_cee56ab6",
  "20260401_0944_caf7b729",
  "20260401_0944_2a740bc9",
  "20260401_0945_69e540dd",
  "20260401_0945_4e397cbc",
  "20260401_0945_43496558",
  "20260401_0946_0caf1eea",
  "20260401_0946_d89385f4",
  "20260401_0947_a5d95fa7",
  "20260401_0947_6bac58bd",
  "20260401_0948_b98bd8d9",
  "20260401_0948_7f2fa323",
  "20260401_0948_ba75886b",
  "20260401_0949_2c5151b9",
  "20260401_0949_5417f92c",
  "20260401_0949_7dff5c66",
  "20260401_0950_e57920c8",
  "20260401_0950_e7696780",
  "20260401_0951_b77b62dd",
  "20260401_0951_a84c5e1a",
  "20260401_0951_733582a5",
  "20260401_0952_d8af2cb0",
  "20260401_0952_8a3a6cc1",
  "20260401_0953_5d1f459f",
  "20260401_0953_64e8f5ad",
  "20260401_0954_4fc3c18f",
  "20260401_0954_ad96f420",
  "20260401_0954_75f70e0e",
  "20260401_0955_6b0ffc0a",
  "20260401_0955_9bee0e22",
  "20260401_0956_d574286a",
  "20260401_0956_de5d59fd",
  "20260401_0956_9790e218",
  "20260401_0957_ff807788",
  "20260401_0957_7266fc88",
  "20260401_0958_a2279157",
  "20260401_0958_cf8d51ff",
  "20260401_0958_4aa43f41",
  "20260401_0959_f288d846",
  "20260401_0959_3a635aa0",
  "20260401_0959_e7f2cf3f",
  "20260401_1000_aecc11e6",
  "20260401_1000_b23f8694",
  "20260401_1001_2c908cd5",
  "20260401_1001_ed4a23f6",
  "20260401_1001_c9083c78",
  "20260401_1002_d3e00aca",
  "20260401_1002_07754880",
  "20260401_1003_5e8d58dc",
  "20260401_1003_fefe905d",
  "20260401_1004_6782cbc9",
  "20260401_1004_dc78557f",
  "20260401_1005_5a308398",
  "20260401_1005_b4fc5ad8",
  "20260401_1006_671f042d",
  "20260401_1006_6fe8e055",
  "20260401_1007_08a9f2fb",
  "20260401_1007_a84e880f",
  "20260401_1007_cfaef121",
  "20260401_1008_d3285cf8",
  "20260401_1008_b72d8e39",
  "20260401_1009_8c118469",
  "20260401_1009_8f6d93cd",
  "20260401_1009_a40b4aee",
  "20260401_1010_ba77da0e",
  "20260401_1010_81c39201",
  "20260401_1011_32a94526",
  "20260401_1011_9dd455fb",
  "20260401_1011_444caf60",
  "20260401_1012_0a1599db",
  "20260401_1012_e94c81df",
  "20260401_1012_79596876",
  "20260401_1013_30eb75a3",
  "20260401_1013_46d1a905",
  "20260401_1014_b0159944",
  "20260401_1014_222f45e9",
  "20260401_1015_089a3ccb",
  "20260401_1015_829de203",
  "20260401_1016_a83323e2",
  "20260401_1016_dc8d0695",
  "20260401_1017_51da46bd",
  "20260401_1017_40ca3fd8",
  "20260401_1017_0fcd6bec",
  "20260401_1018_1d061ccb",
  "20260401_1018_57e88429",
  "20260401_1019_b9dcc666",
  "20260401_1019_a80b322a",
  "20260401_1019_ea637b44",
  "20260401_1020_9d703696",
  "20260401_1020_10361cd1",
  "20260401_1021_f79f57b9",
  "20260401_1021_cb221e4b",
  "20260401_1021_70c9d9a4",
  "20260401_1022_292b560c",
  "20260401_1022_f4d1278e",
  "20260401_1023_5f6a6dc5",
  "20260401_1023_a37e0c3f",
  "20260401_1023_696c4956",
  "20260401_1024_4bf93817",
  "20260401_1024_4b0f86fe"
]

In [12]:
CONFIG = {
    # Paths
    "artifacts_root": str(Path.cwd().parent.parent / "artifacts"),
    "artifact_group": "lee-2019-fine-tuning",
    "artifact_paradigm": "SSVEP",

    # Runs to evaluate
    "run_ids": RUN_IDS,

    # Ranking and reporting
    "primary_metric": "mean_accuracy",  # mean_accuracy or mean_balanced_accuracy
    "top_k": 10,
    "show_config_columns": ["seed", "learning_rate", "batch_size", "subjects_to_use"],

    # Subject-focused analysis
    "focus_subject": "51",  # str/int subject id, or None to disable
    "show_focus_subject_folds": True,

    # Optional outputs
    "show_run_summary_table": False,
    "show_subject_long_table": False,
    "show_fold_long_table": False,
    "export_tables": True,
}

In [13]:
ARTIFACTS_ROOT = Path(CONFIG["artifacts_root"])
RUN_IDS = CONFIG["run_ids"]
GROUP_DIR = ARTIFACTS_ROOT / CONFIG["artifact_group"] / CONFIG["artifact_paradigm"]

assert ARTIFACTS_ROOT.exists(), f"Artifacts root does not exist: {ARTIFACTS_ROOT}"
assert GROUP_DIR.exists(), f"Artifact group does not exist: {GROUP_DIR}"
assert RUN_IDS, "CONFIG['run_ids'] cannot be empty"

print("Artifacts root:", ARTIFACTS_ROOT)
print("Artifact group:", GROUP_DIR)
print("Run count:", len(RUN_IDS))
print("Primary metric:", CONFIG["primary_metric"])

Artifacts root: /home/vegorov/Repos/eeg_jepa_research/artifacts
Artifact group: /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/SSVEP
Run count: 100
Primary metric: mean_accuracy


## 2. Helpers

These helpers build compact tables for ranking and optional subject/fold deep dives.

In [14]:
def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def flatten_dict(d: dict[str, Any], parent_key: str = "", sep: str = ".") -> dict[str, Any]:
    out: dict[str, Any] = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)
        if isinstance(v, dict):
            out.update(flatten_dict(v, new_key, sep=sep))
        else:
            out[new_key] = v
    return out


def load_run_artifacts(group_dir: Path, run_id: str) -> dict[str, Any] | None:
    run_dir = group_dir / run_id
    if not run_dir.exists():
        print(f"  [SKIP] Run folder not found: {run_id}")
        return None

    files = {
        "config": run_dir / "config.json",
        "run_metadata": run_dir / "run_metadata.json",
        "global_metrics": run_dir / "global_metrics.json",
        "subject_metrics": run_dir / "subject_metrics.json",
        "cv_results": run_dir / "cv_results.json",
    }

    missing = [name for name, path in files.items() if not path.exists()]
    if missing:
        print(f"  [SKIP] Run {run_id} is missing artifact files: {missing}")
        return None

    return {
        "run_id": run_id,
        "run_dir": run_dir,
        "config": read_json(files["config"]),
        "run_metadata": read_json(files["run_metadata"]),
        "global_metrics": read_json(files["global_metrics"]),
        "subject_metrics": read_json(files["subject_metrics"]),
        "cv_results": read_json(files["cv_results"]),
    }


def build_run_summary_df(runs: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for run in runs:
        row = {
            "run_id": run["run_id"],
            "run_dir": str(run["run_dir"]),
        }
        row.update({f"config.{k}": v for k, v in flatten_dict(run["config"]).items()})
        row.update({f"meta.{k}": v for k, v in flatten_dict(run["run_metadata"]).items()})
        row.update({f"global.{k}": v for k, v in flatten_dict(run["global_metrics"]).items()})
        rows.append(row)
    return pd.DataFrame(rows)


def build_subject_df(runs: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for run in runs:
        for subject_id, metrics in run["subject_metrics"].items():
            row = {
                "run_id": run["run_id"],
                "subject_id": str(subject_id),
            }
            row.update(flatten_dict(metrics))
            rows.append(row)
    return pd.DataFrame(rows)


def build_fold_df(runs: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for run in runs:
        for fold in run["cv_results"]:
            row = {"run_id": run["run_id"]}
            row.update(fold)
            row["subject_id"] = str(row["subject_id"])
            rows.append(row)
    return pd.DataFrame(rows)


def order_by_run_ids(df: pd.DataFrame, run_ids: list[str]) -> pd.DataFrame:
    if "run_id" not in df.columns:
        return df
    out = df.copy()
    out["run_id"] = pd.Categorical(out["run_id"], categories=run_ids, ordered=True)
    out = out.sort_values("run_id").reset_index(drop=True)
    out["run_id"] = out["run_id"].astype(str)
    return out

## 3. Load and Build Core Tables

In [15]:
_raw = [load_run_artifacts(GROUP_DIR, run_id) for run_id in RUN_IDS]
runs = [r for r in _raw if r is not None]
_skipped = len(_raw) - len(runs)
if _skipped:
    print(f"Skipped {_skipped} run(s) (not yet on disk or incomplete).")

assert runs, "No runs were loaded. Check that run_ids exist under GROUP_DIR."

loaded_ids = [r["run_id"] for r in runs]
run_summary_df = order_by_run_ids(build_run_summary_df(runs), loaded_ids)
subject_df     = order_by_run_ids(build_subject_df(runs), loaded_ids)
fold_df        = order_by_run_ids(build_fold_df(runs), loaded_ids)

print(f"Loaded {len(runs)} / {len(RUN_IDS)} runs")
print(f"run_summary_df: {run_summary_df.shape}")
print(f"subject_df:     {subject_df.shape}")
print(f"fold_df:        {fold_df.shape}")

Loaded 100 / 100 runs
run_summary_df: (100, 38)
subject_df:     (100, 8)
fold_df:        (500, 14)


## 4. Overall Leaderboard (Best Runs/Configs)

This is the main output for comparing runs.

In [16]:
metric = CONFIG["primary_metric"]
metric_col = f"global.{metric}"
assert metric_col in run_summary_df.columns, f"Missing metric column: {metric_col}"

config_cols = [f"config.{k}" for k in CONFIG["show_config_columns"] if f"config.{k}" in run_summary_df.columns]

# Build leaderboard columns, removing duplicates while preserving order
primary_cols = [c for c in ["run_id", metric_col, "global.mean_accuracy", "global.mean_balanced_accuracy"] if c in run_summary_df.columns]
leaderboard_cols = list(dict.fromkeys(primary_cols + config_cols))

leaderboard_df = run_summary_df[leaderboard_cols].sort_values(metric_col, ascending=False).reset_index(drop=True)
top_k = min(CONFIG["top_k"], len(leaderboard_df))

print(f"Top {top_k} runs by {metric_col}:")
display(leaderboard_df.head(top_k))

if CONFIG["show_run_summary_table"]:
    display(run_summary_df)

Top 10 runs by global.mean_accuracy:


,run_id,global.mean_accuracy,global.mean_balanced_accuracy,config.seed,config.learning_rate,config.batch_size,config.subjects_to_use
0,20260401_1004_dc78557f,0.40,0.40,3008,0.0010,24,[51]
1,20260401_1015_089a3ccb,0.39,0.39,4013,0.0010,16,[51]
2,20260401_1001_ed4a23f6,0.38,0.38,3001,0.0010,8,[51]
3,20260401_0954_75f70e0e,0.37,0.37,2008,0.0010,32,[51]
4,20260401_1024_4bf93817,0.37,0.37,4035,0.0010,8,[51]
5,20260401_1016_dc8d0695,0.36,0.36,4016,0.0010,16,[51]
6,20260401_1001_2c908cd5,0.35,0.35,2024,0.0025,32,[51]
7,20260401_0946_0caf1eea,0.35,0.35,1007,0.0010,16,[51]
8,20260401_0944_2a740bc9,0.34,0.34,1003,0.0010,16,[51]
9,20260401_1005_b4fc5ad8,0.34,0.34,3010,0.0010,24,[51]


## 5. Focus Subject Evaluation (Optional)

Set `focus_subject` in `CONFIG` to rank runs for one subject and inspect fold-level details.

In [17]:
focus_subject = CONFIG["focus_subject"]

if focus_subject is None:
    print("Focus subject analysis is disabled (focus_subject=None).")
else:
    focus_subject = str(focus_subject)
    subject_metric_col = "mean_accuracy"
    if CONFIG["primary_metric"] == "mean_balanced_accuracy" and "mean_balanced_accuracy" in subject_df.columns:
        subject_metric_col = "mean_balanced_accuracy"

    focus_subject_df = subject_df[subject_df["subject_id"] == focus_subject].copy()
    if focus_subject_df.empty:
        print(f"No subject metrics found for subject_id={focus_subject}")
    else:
        focus_subject_df = focus_subject_df.sort_values(subject_metric_col, ascending=False).reset_index(drop=True)
        top_k = min(CONFIG["top_k"], len(focus_subject_df))
        print(f"Top {top_k} runs for subject {focus_subject} by {subject_metric_col}:")
        display(focus_subject_df[[c for c in ["run_id", "subject_id", "mean_accuracy", "mean_balanced_accuracy", "std_accuracy"] if c in focus_subject_df.columns]].head(top_k))

        if CONFIG["show_focus_subject_folds"]:
            focus_folds = fold_df[fold_df["subject_id"] == focus_subject].copy()
            if focus_folds.empty:
                print(f"No fold-level rows found for subject_id={focus_subject}")
            else:
                fold_metric_col = "accuracy" if CONFIG["primary_metric"] == "mean_accuracy" else "balanced_accuracy"
                fold_metric_col = fold_metric_col if fold_metric_col in focus_folds.columns else "accuracy"
                focus_folds = focus_folds.sort_values(["run_id", fold_metric_col], ascending=[True, False]).reset_index(drop=True)
                print(f"Fold details for subject {focus_subject}:")
                display(focus_folds[[c for c in ["run_id", "subject_id", "fold_id", "accuracy", "balanced_accuracy", "best_valid_loss"] if c in focus_folds.columns]])

if CONFIG["show_subject_long_table"]:
    display(subject_df)

if CONFIG["show_fold_long_table"]:
    display(fold_df)

Top 10 runs for subject 51 by mean_accuracy:


,run_id,subject_id,mean_accuracy,mean_balanced_accuracy,std_accuracy
0,20260401_1004_dc78557f,51,0.40,0.40,0.100000
1,20260401_1015_089a3ccb,51,0.39,0.39,0.080000
2,20260401_1001_ed4a23f6,51,0.38,0.38,0.097980
3,20260401_0954_75f70e0e,51,0.37,0.37,0.124900
4,20260401_1024_4bf93817,51,0.37,0.37,0.092736
5,20260401_1016_dc8d0695,51,0.36,0.36,0.066332
6,20260401_1001_2c908cd5,51,0.35,0.35,0.031623
7,20260401_0946_0caf1eea,51,0.35,0.35,0.094868
8,20260401_0944_2a740bc9,51,0.34,0.34,0.058310
9,20260401_1005_b4fc5ad8,51,0.34,0.34,0.080000


Fold details for subject 51:


,run_id,subject_id,fold_id,accuracy,balanced_accuracy,best_valid_loss
0,20260401_0944_2a740bc9,51,2,0.40,0.40,1.362463
1,20260401_0944_2a740bc9,51,3,0.40,0.40,1.358874
2,20260401_0944_2a740bc9,51,5,0.35,0.35,1.300765
3,20260401_0944_2a740bc9,51,1,0.30,0.30,1.301931
4,20260401_0944_2a740bc9,51,4,0.25,0.25,1.385623
...,...,...,...,...,...,...
495,20260401_1024_4bf93817,51,4,0.50,0.50,1.385656
496,20260401_1024_4bf93817,51,5,0.45,0.45,1.199724
497,20260401_1024_4bf93817,51,2,0.35,0.35,1.301958
498,20260401_1024_4bf93817,51,1,0.30,0.30,1.363408


## 6. Export (Optional)

Exports compact tables so you can review top runs/configs outside the notebook.

In [18]:
if CONFIG["export_tables"]:
    export_dir = GROUP_DIR / "_comparison_exports"
    export_dir.mkdir(exist_ok=True)

    leaderboard_df.to_csv(export_dir / "leaderboard.csv", index=False)
    run_summary_df.to_csv(export_dir / "run_summary_full.csv", index=False)
    subject_df.to_csv(export_dir / "subject_summary_long.csv", index=False)
    fold_df.to_csv(export_dir / "fold_summary_long.csv", index=False)

    if CONFIG["focus_subject"] is not None:
        s = str(CONFIG["focus_subject"])
        focus_subject_df = subject_df[subject_df["subject_id"] == s]
        focus_folds_df = fold_df[fold_df["subject_id"] == s]
        focus_subject_df.to_csv(export_dir / f"subject_{s}_summary.csv", index=False)
        focus_folds_df.to_csv(export_dir / f"subject_{s}_folds.csv", index=False)

    print("Exported tables to:", export_dir)
else:
    print("Export is disabled in CONFIG.")

Exported tables to: /home/vegorov/Repos/eeg_jepa_research/artifacts/lee-2019-fine-tuning/SSVEP/_comparison_exports
